# EcoGraphRAG — Notebook 3: BM25 Baseline

**Goal:** Run 500 HotpotQA questions through BM25 retrieval + Mistral-7B 4-bit.

**Inputs:** `chunks.json`, `hotpotqa_500.json`

**Outputs:** `bm25_results.json`

## 1. Setup

In [ ]:
from google.colab import drive
import os, json, time, re

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/graphrag_research/'
DRIVE_DATA = DRIVE_ROOT + 'data/'
DRIVE_OUTPUTS = DRIVE_ROOT + 'outputs/'
DRIVE_CHECKPOINTS = DRIVE_ROOT + 'checkpoints/'

for d in [DRIVE_DATA, DRIVE_OUTPUTS, DRIVE_CHECKPOINTS]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')

In [ ]:
!pip install -q transformers accelerate bitsandbytes rank-bm25

## 2. Load Data

In [ ]:
import torch
import numpy as np

with open(DRIVE_DATA + 'hotpotqa_500.json', 'r') as f:
    dataset = json.load(f)
questions = dataset['questions']
print(f'Loaded {len(questions)} questions')

with open(DRIVE_DATA + 'chunks.json', 'r') as f:
    chunks = json.load(f)
print(f'Loaded {len(chunks)} chunks')

## 3. Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

BM25_TOP_K = 5

def tokenize_bm25(text):
    return [t for t in re.findall(r'[a-z0-9]+', text.lower()) if len(t) > 1]

print('Building BM25 index...')
tokenized = [tokenize_bm25(c['text']) for c in chunks]
bm25 = BM25Okapi(tokenized)
print(f'BM25 index built: {len(tokenized)} documents')

def bm25_retrieve(query, top_k=BM25_TOP_K):
    scores = bm25.get_scores(tokenize_bm25(query))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [chunks[i] for i in top_idx]

# Quick test
test = bm25_retrieve('Scott Derrickson nationality')
print(f'Test: {test[0]["title"]} — {test[0]["text"][:80]}...')

## 4. Load Mistral-7B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'
LLM_MAX_NEW_TOKENS = 64

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, quantization_config=bnb_config,
    device_map='auto', torch_dtype=torch.float16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Model loaded. GPU: {torch.cuda.max_memory_allocated()/(1024**2):.0f} MB')

In [ ]:
PROMPT_TEMPLATE = """<s>[INST] Answer the following question using ONLY the provided context.
Give a short, precise answer (1-5 words). Do not explain.

Context:
{context}

Question: {question}

Answer: [/INST]"""

def generate_answer(question, context_chunks):
    context = '\n\n'.join([c['text'] for c in context_chunks])
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=LLM_MAX_NEW_TOKENS,
                                 do_sample=False, temperature=0.1,
                                 pad_token_id=tokenizer.pad_token_id)
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 5. Run BM25 Baseline (500 Questions)

In [ ]:
import tracemalloc

CHECKPOINT_FILE = DRIVE_CHECKPOINTS + 'bm25_checkpoint.json'
RESULTS_FILE = DRIVE_OUTPUTS + 'bm25_results.json'
CHECKPOINT_INTERVAL = 25

# Load checkpoint if exists
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        ckpt = json.load(f)
    results = ckpt['results']
    start_idx = ckpt['current_index'] + 1
    print(f'Resuming from checkpoint: {len(results)} results, index {start_idx}')
else:
    results = []
    start_idx = 0

total = len(questions)
print(f'Running BM25 baseline: questions {start_idx} to {total - 1}...\n')

for i in range(start_idx, total):
    q = questions[i]
    torch.cuda.reset_peak_memory_stats()
    tracemalloc.start()
    t_start = time.time()

    retrieved = bm25_retrieve(q['question'])
    prediction = generate_answer(q['question'], retrieved)

    latency = time.time() - t_start
    gpu_mb = torch.cuda.max_memory_allocated() / (1024**2)
    _, ram_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    results.append({
        'id': q['id'], 'question': q['question'],
        'prediction': prediction, 'gold': q['answer'],
        'type': q['type'], 'level': q.get('level', 'unknown'),
        'latency_seconds': round(latency, 3),
        'peak_gpu_mb': round(gpu_mb, 2),
        'peak_ram_mb': round(ram_peak / (1024**2), 2),
        'chunks_retrieved': len(retrieved),
    })

    if (i + 1) % 10 == 0 or i == total - 1:
        print(f'  [{i+1}/{total}] {latency:.2f}s  Pred:"{prediction[:40]}"  Gold:"{q["answer"]}"')

    if (i + 1) % CHECKPOINT_INTERVAL == 0:
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({'current_index': i, 'results': results}, f, ensure_ascii=False)
        print(f'  \U0001f4be Checkpoint saved at index {i}')

with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'\n\u2705 Saved {len(results)} results to {RESULTS_FILE}')

if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

## 6. Evaluation

In [ ]:
import string
from collections import Counter

def normalize(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(c for c in s if c not in string.punctuation)
    return ' '.join(s.split())

def em(p, g): return int(normalize(p) == normalize(g))

def f1(p, g):
    pt, gt = normalize(p).split(), normalize(g).split()
    if not pt or not gt: return float(pt == gt)
    c = sum((Counter(pt) & Counter(gt)).values())
    if c == 0: return 0.0
    return 2 * (c/len(pt)) * (c/len(gt)) / (c/len(pt) + c/len(gt))

em_scores = [em(r['prediction'], r['gold']) for r in results]
f1_scores = [f1(r['prediction'], r['gold']) for r in results]

print(f'\n{"="*60}')
print(f'BM25 BASELINE RESULTS ({len(results)} questions)')
print(f'{"="*60}')
print(f'Overall EM:  {sum(em_scores)/len(em_scores):.4f}  ({sum(em_scores)/len(em_scores)*100:.1f}%)')
print(f'Overall F1:  {sum(f1_scores)/len(f1_scores):.4f}  ({sum(f1_scores)/len(f1_scores)*100:.1f}%)')

for qt in ['bridge', 'comparison']:
    tr = [r for r in results if r['type'] == qt]
    if tr:
        te, tf = [em(r['prediction'],r['gold']) for r in tr], [f1(r['prediction'],r['gold']) for r in tr]
        print(f'\n{qt.capitalize()} ({len(tr)} Qs):  EM: {sum(te)/len(te):.4f}  F1: {sum(tf)/len(tf):.4f}')

latencies = [r['latency_seconds'] for r in results]
print(f'\nLatency: avg={sum(latencies)/len(latencies):.2f}s  median={sorted(latencies)[len(latencies)//2]:.2f}s')

metrics = {'experiment': 'bm25_baseline', 'num_questions': len(results),
           'overall_em': round(sum(em_scores)/len(em_scores),4),
           'overall_f1': round(sum(f1_scores)/len(f1_scores),4),
           'avg_latency_s': round(sum(latencies)/len(latencies),3)}
with open(DRIVE_OUTPUTS + 'bm25_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nMetrics saved. \u2705 BM25 baseline complete!')